<p align="center"><a href="https://www.plus2net.com/python/pandas-add_prefix.php"><img src="https://www.plus2net.com/images/top2.jpg" width="300" alt="plus2net Python tutorials"></a></p>

# Pandas add_prefix(): Name Columns and Index Labels

Run cells in order in Colab. All sample data are synthetic and embedded. Save a copy in Drive to keep edits. Try the exercise before its solution. Explicit axis selection requires Pandas 2.0 or newer.

## Prefix DataFrame column labels

For a DataFrame, add_prefix() prefixes column labels by default. Col_ changes NAME to Col_NAME, not to name1. It changes labels rather than values and returns a new DataFrame.

In [ ]:
import pandas as pd
df = pd.DataFrame({"NAME": ["Ravi", "Raju", "Alex", "Ron", "King", "Jack"], "ID": [1, 2, 3, 4, 5, 6], "MATH": [30, 40, 50, 60, 70, 80], "ENGLISH": [20, 30, 40, 50, 60, 70]})
result = df.add_prefix("Col_")
print(result)
assert result.columns.tolist() == ["Col_NAME", "Col_ID", "Col_MATH", "Col_ENGLISH"]
assert df.columns.tolist() == ["NAME", "ID", "MATH", "ENGLISH"]

**Expected output**

```text
  Col_NAME  Col_ID  Col_MATH  Col_ENGLISH
0     Ravi       1        30           20
1     Raju       2        40           30
2     Alex       3        50           40
3      Ron       4        60           50
4     King       5        70           60
5     Jack       6        80           70
```

## Prefix Series index labels

For a Series, the row index labels are prefixed. This does not prepend text to the stored values or change the Series name.

In [ ]:
s = pd.Series([1, 5, 7, 9, 10], name="score")
result = s.add_prefix("Ser_")
print(result)
assert result.index.tolist() == ["Ser_0", "Ser_1", "Ser_2", "Ser_3", "Ser_4"]
assert result.tolist() == s.tolist() and result.name == "score"

**Expected output**

```text
Ser_0     1
Ser_1     5
Ser_2     7
Ser_3     9
Ser_4    10
Name: score, dtype: int64
```

## Choose columns or row labels explicitly

axis="columns" prefixes columns and axis="index" prefixes rows for a DataFrame. The axis parameter is available from Pandas 2.0. Prefixing the index changes label-based selection and alignment.

In [ ]:
table = pd.DataFrame({"amount": [12, 20]}, index=[101, 102])
print(table.add_prefix("sales_", axis="columns"))
print(table.add_prefix("order_", axis="index"))
assert table.add_prefix("order_", axis="index").index.tolist() == ["order_101", "order_102"]

**Expected output**

```text
     sales_amount
101            12
102            20
           amount
order_101      12
order_102      20
```

## Do not confuse label prefixes with value prefixes

To prefix values, operate on the column itself. Use a nullable string dtype to preserve missing values. For combining fields, see <a href="https://www.plus2net.com/python/pandas-str-cat.php">str.cat()</a>.

In [ ]:
table = pd.DataFrame({"code": pd.Series(["0012", None], dtype="string")})
renamed = table.add_prefix("source_")
table["display_code"] = "SKU-" + table["code"]
print(renamed)
print(table)
assert renamed["source_code"].iloc[0] == "0012"
assert table["display_code"].iloc[0] == "SKU-0012"
assert pd.isna(table["display_code"].iloc[1])

**Expected output**

```text
  source_code
0        0012
1        <NA>
   code display_code
0  0012     SKU-0012
1  <NA>         <NA>
```

## Prefix selected fields while keeping a shared key

add_prefix() applies to the whole selected axis. Use rename() with a mapping when a join key must retain its name. Preserve one shared customer_id field and prefix the payload columns.

In [ ]:
sales = pd.DataFrame({"customer_id": [101, 102], "amount": [12.5, 20], "status": ["paid", "paid"]})
mapping = {column: "sales_" + column for column in sales.columns if column != "customer_id"}
prepared = sales.rename(columns=mapping)
print(prepared)
assert prepared.columns.tolist() == ["customer_id", "sales_amount", "sales_status"]

**Expected output**

```text
   customer_id  sales_amount sales_status
0          101          12.5         paid
1          102          20.0         paid
```

## Validate labels before and after prefixing

Prefixing does not repair duplicate labels. It also converts labels to string representations, which can collapse originally different labels such as integer 1 and string "1". Inspect <a href="https://www.plus2net.com/python/pandas-dataframe-columns.php">columns</a> before combining datasets.

In [ ]:
table = pd.DataFrame([[5, 7]], columns=[1, "1"])
result = table.add_prefix("source_")
print(result)
print("Input labels unique:", table.columns.is_unique)
print("Output labels unique:", result.columns.is_unique)
assert table.columns.is_unique and not result.columns.is_unique

**Expected output**

```text
   source_1  source_1
0         5         7
Input labels unique: True
Output labels unique: False
```

## Practical workflow: combine named source tables

This synthetic example joins one customer table and one order-summary table with one row per customer. Prefix each source payload while keeping the common key intact. Validate key uniqueness before merging; naming columns does not validate the relationship.

In [ ]:
customers = pd.DataFrame({"customer_id": [101, 102], "name": ["Ravi", "Alex"], "status": ["active", "inactive"]})
summary = pd.DataFrame({"customer_id": [101, 102], "amount": [12.5, 20], "status": ["paid", "pending"]})
def prefix_payload(table, prefix, key):
    assert table.columns.is_unique
    assert table[key].notna().all() and table[key].is_unique
    mapping = {column: prefix + str(column) for column in table.columns if column != key}
    result = table.rename(columns=mapping)
    assert result.columns.is_unique
    return result
left = prefix_payload(customers, "customer_", "customer_id")
right = prefix_payload(summary, "sales_", "customer_id")
combined = left.merge(right, on="customer_id", how="left", validate="one_to_one", indicator=True)
print(combined)
assert combined.columns.is_unique and len(combined) == 2
assert combined["_merge"].eq("both").all()

**Expected output**

```text
   customer_id customer_name customer_status  sales_amount sales_status _merge
0          101          Ravi          active          12.5         paid   both
1          102          Alex        inactive          20.0      pending   both
```

## Check the output schema and retain the naming map

Confirm that source-specific statuses remain separate and keys are preserved. The merge indicator exposes unmatched customers. Keep naming maps when downstream reports depend on a stable schema.

In [ ]:
expected = ["customer_id", "customer_name", "customer_status", "sales_amount", "sales_status", "_merge"]
assert combined.columns.tolist() == expected
assert combined["customer_status"].tolist() == ["active", "inactive"]
assert combined["sales_status"].tolist() == ["paid", "pending"]
print(combined.drop(columns="_merge").to_csv(index=False).rstrip())
# Optional Colab export:
# combined.drop(columns="_merge").to_csv("named_customer_sales.csv", index=False)

**Expected output**

```text
customer_id,customer_name,customer_status,sales_amount,sales_status
101,Ravi,active,12.5,paid
102,Alex,inactive,20.0,pending
```

## Avoid adding the same prefix twice unintentionally

Repeated calls add repeated prefixes. If your process receives both prefixed and unprefixed labels, define a naming rule and check for collisions after conditional renaming. Do not blindly strip prefixes that may be meaningful.

In [ ]:
table = pd.DataFrame({"amount": [5], "sales_status": ["paid"]})
mapping = {column: column if column.startswith("sales_") else "sales_" + column for column in table.columns}
result = table.rename(columns=mapping)
assert result.columns.is_unique
print(result)
print("Repeated prefix on amount:", table[["amount"]].add_prefix("sales_").add_prefix("sales_").columns.tolist())

**Expected output**

```text
   sales_amount sales_status
0             5         paid
Repeated prefix on amount: ['sales_sales_amount']
```

## Choose add_prefix(), add_suffix() or rename()

Use add_prefix() for an entire axis, <a href="https://www.plus2net.com/python/pandas-add_suffix.php">add_suffix()</a> for trailing label text, and <a href="https://www.plus2net.com/python/pandas-rename.php">rename()</a> for selective or arbitrary label changes. For row labels, consider whether changing them will disrupt <a href="https://www.plus2net.com/python/pandas-dataframe-loc.php">loc selection</a> or alignment. These methods organise labels; they do not clean the stored values.

## Common questions

<strong>Does add_prefix() change cell values?</strong> No. <strong>Is there inplace=True?</strong> No; use the returned object. <strong>Why did my numeric index become text?</strong> The prefix forms string labels. <strong>Can I prefix only two columns?</strong> Use a rename mapping or select and combine deliberately. <strong>Does prefixing fix duplicate columns?</strong> No; check label uniqueness. <strong>Should I prefix join keys?</strong> Preserve the shared key or explicitly adapt the merge arguments. <strong>Does a prefix guarantee compatible data?</strong> No; still validate types and business relationships.

## Exercise: prefix only measurement fields

Create a table with id, math and english. Keep id unchanged and rename the two measurement fields to exam_math and exam_english. Check that all values and the original table remain unchanged.

## Exercise solution

Use an explicit mapping to leave the identifier untouched.

In [ ]:
practice = pd.DataFrame({"id": [1, 2], "math": [75, 85], "english": [70, 80]})
answer = practice.rename(columns={"math": "exam_math", "english": "exam_english"})
print(answer)
assert answer.columns.tolist() == ["id", "exam_math", "exam_english"]
assert practice.columns.tolist() == ["id", "math", "english"]
assert answer["exam_math"].equals(practice["math"])

**Expected output**

```text
   id  exam_math  exam_english
0   1         75            70
1   2         85            80
```

## Practice in Google Colab

<a class="btn btn-outline-primary" href="https://colab.research.google.com/github/plus2net/pandas/blob/main/pandas_add_prefix_practice.ipynb" target="_blank" rel="noopener">Open in Google Colab</a> <a class="btn btn-outline-secondary" href="https://github.com/plus2net/pandas/blob/main/pandas_add_prefix_practice.ipynb" target="_blank" rel="noopener">View on GitHub</a><br>Run the examples in order, change prefixes and selected fields, then inspect the output labels and preserved keys. Save your own copy to keep edits. All sample tables are included in the notebook.